In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
file_path = '/content/Student_Wellbeing_Survey.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Dataset loaded successfully with shape: {df.shape}")
except FileNotFoundError:
    print("Please ensure the CSV file is uploaded and the path is correct.")

Dataset loaded successfully with shape: (600, 20)


In [3]:
cols = ['Weekly_Study_Hours', 'Average_Sleep_Hours', 'Daily_Screen_Time_Hours',
        'Stress_Score', 'Academic_Readiness_Score']

print("--- Central Tendency ---")
for col in cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    mode_val = df[col].mode()[0]

    print(f"\n{col}:")
    print(f"  Mean   : {mean_val:.2f}")
    print(f"  Median : {median_val:.2f}")
    print(f"  Mode   : {mode_val:.2f}")

--- Central Tendency ---

Weekly_Study_Hours:
  Mean   : 15.69
  Median : 15.40
  Mode   : 13.10

Average_Sleep_Hours:
  Mean   : 7.00
  Median : 7.00
  Mode   : 7.00

Daily_Screen_Time_Hours:
  Mean   : 4.50
  Median : 4.20
  Mode   : 3.50

Stress_Score:
  Mean   : 4.46
  Median : 4.50
  Mode   : 4.70

Academic_Readiness_Score:
  Mean   : 71.77
  Median : 71.65
  Mode   : 71.10


In [4]:
print("--- Measures of Dispersion ---")
dispersion_data = []

for col in cols:
    range_val = df[col].max() - df[col].min()
    var_val = df[col].var()
    std_val = df[col].std()
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    dispersion_data.append({
        'Variable': col, 'Range': range_val, 'Variance': var_val,
        'Std_Dev': std_val, 'Q1': q1, 'Q3': q3, 'IQR': iqr
    })

disp_df = pd.DataFrame(dispersion_data).set_index('Variable')
display(disp_df.round(2))

max_var_col = disp_df['Std_Dev'].idxmax()
print(f"\n>> The variable with the greatest variability (highest Std Dev) is: {max_var_col}")

--- Measures of Dispersion ---


,Range,Variance,Std_Dev,Q1,Q3,IQR
Variable,,,,,,
Weekly_Study_Hours,30.0,19.22,4.38,13.00,18.42,5.42
Average_Sleep_Hours,5.0,0.70,0.84,6.48,7.60,1.12
Daily_Screen_Time_Hours,11.2,3.47,1.86,3.20,5.40,2.20
Stress_Score,8.4,2.78,1.67,3.30,5.70,2.40
Academic_Readiness_Score,56.1,93.98,9.69,65.20,78.03,12.83



>> The variable with the greatest variability (highest Std Dev) is: Academic_Readiness_Score


In [5]:
outlier_cols = ['Weekly_Study_Hours', 'Daily_Screen_Time_Hours',
                'Commute_Time_Minutes', 'Monthly_Discretionary_Spending']

def remove_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]
    clean_df = df[(df[col] >= lower) & (df[col] <= upper)]
    return clean_df, len(outliers)

print("--- Outlier Detection ---")
# Let's focus our before/after comparison on 'Monthly_Discretionary_Spending'
col_to_compare = 'Monthly_Discretionary_Spending'

orig_mean = df[col_to_compare].mean()
orig_med = df[col_to_compare].median()

clean_df, num_outliers = remove_outliers(df, col_to_compare)

clean_mean = clean_df[col_to_compare].mean()
clean_med = clean_df[col_to_compare].median()

print(f"{col_to_compare}: Found {num_outliers} outliers.")
print(f"BEFORE -> Mean: {orig_mean:.2f} | Median: {orig_med:.2f}")
print(f"AFTER  -> Mean: {clean_mean:.2f} | Median: {clean_med:.2f}")

--- Outlier Detection ---
Monthly_Discretionary_Spending: Found 20 outliers.
BEFORE -> Mean: 6343.78 | Median: 5685.50
AFTER  -> Mean: 5973.72 | Median: 5590.50


In [6]:
N = len(df)

# Define boolean masks for the events
A_mask = df['Part_Time_Job'] == 'Yes'
B_mask = df['Stress_Score'] >= 7
C_mask = df['Scholarship'] == 'Yes'
D_mask = df['Exercise_Days_Per_Week'] >= 3

# Calculate basic probabilities
P_A = A_mask.sum() / N
P_B = B_mask.sum() / N
P_C = C_mask.sum() / N
P_D = D_mask.sum() / N

# Compound events
P_A_and_B = (A_mask & B_mask).sum() / N
P_A_or_B = (A_mask | B_mask).sum() / N

# Conditional probabilities: P(A|B) = P(A and B) / P(B)
P_A_given_B = P_A_and_B / P_B
P_B_given_A = P_A_and_B / P_A

print("--- Probability Calculations ---")
print(f"P(A) [Part-time Job]     = {P_A:.4f}")
print(f"P(B) [Stress >= 7]       = {P_B:.4f}")
print(f"P(C) [Scholarship]       = {P_C:.4f}")
print(f"P(D) [Exercise >= 3]     = {P_D:.4f}")
print(f"P(A or B)                = {P_A_or_B:.4f}")
print(f"P(A and B)               = {P_A_and_B:.4f}")
print(f"P(A | B)                 = {P_A_given_B:.4f}")
print(f"P(B | A)                 = {P_B_given_A:.4f}")

--- Probability Calculations ---
P(A) [Part-time Job]     = 0.2533
P(B) [Stress >= 7]       = 0.0750
P(C) [Scholarship]       = 0.3067
P(D) [Exercise >= 3]     = 0.5933
P(A or B)                = 0.2767
P(A and B)               = 0.0517
P(A | B)                 = 0.6889
P(B | A)                 = 0.2039


In [7]:
print("--- Checking Independence ---")
P_A_times_P_B = P_A * P_B

print(f"P(A and B)    = {P_A_and_B:.4f}")
print(f"P(A) * P(B)   = {P_A_times_P_B:.4f}")

if abs(P_A_and_B - P_A_times_P_B) < 0.001:
    print("Conclusion: The events APPEAR INDEPENDENT (values are practically equal).")
else:
    print("Conclusion: The events are DEPENDENT (values are not equal).")

--- Checking Independence ---
P(A and B)    = 0.0517
P(A) * P(B)   = 0.0190
Conclusion: The events are DEPENDENT (values are not equal).


In [8]:
print("--- Bayes' Theorem ---")
# Need P(not A) and P(B | not A)
not_A_mask = ~A_mask
P_not_A = not_A_mask.sum() / N
P_B_given_not_A = (not_A_mask & B_mask).sum() / not_A_mask.sum()

# Apply Bayes' Theorem
numerator = P_B_given_A * P_A
denominator = (P_B_given_A * P_A) + (P_B_given_not_A * P_not_A)
bayes_P_A_given_B = numerator / denominator

print(f"Calculated using Bayes' formula : {bayes_P_A_given_B:.4f}")
print(f"Direct conditional calculation  : {P_A_given_B:.4f}")
print("Verification: Both values match exactly!")

--- Bayes' Theorem ---
Calculated using Bayes' formula : 0.6889
Direct conditional calculation  : 0.6889
Verification: Both values match exactly!


In [9]:
print("--- Normal Distribution & Z-Scores (Academic Readiness) ---")
readiness_mean = df['Academic_Readiness_Score'].mean()
readiness_std = df['Academic_Readiness_Score'].std()

highest_score = df['Academic_Readiness_Score'].max()
lowest_score = df['Academic_Readiness_Score'].min()

z_high = (highest_score - readiness_mean) / readiness_std
z_low = (lowest_score - readiness_mean) / readiness_std

print(f"Mean: {readiness_mean:.2f}, Std Dev: {readiness_std:.2f}")
print(f"Highest Score: {highest_score:.2f} | Z-Score: {z_high:.2f}")
print(f"Lowest Score : {lowest_score:.2f} | Z-Score: {z_low:.2f}")

# Empirical Rule (68-95-99.7)
print("\n--- Empirical Rule (68-95-99.7) ---")
print(f"~68% of students score between: {readiness_mean - readiness_std:.2f} and {readiness_mean + readiness_std:.2f} (1 Std Dev)")
print(f"~95% of students score between: {readiness_mean - 2*readiness_std:.2f} and {readiness_mean + 2*readiness_std:.2f} (2 Std Devs)")
print(f"~99.7% of students score between: {readiness_mean - 3*readiness_std:.2f} and {readiness_mean + 3*readiness_std:.2f} (3 Std Devs)")

--- Normal Distribution & Z-Scores (Academic Readiness) ---
Mean: 71.77, Std Dev: 9.69
Highest Score: 98.00 | Z-Score: 2.71
Lowest Score : 41.90 | Z-Score: -3.08

--- Empirical Rule (68-95-99.7) ---
~68% of students score between: 62.08 and 81.47 (1 Std Dev)
~95% of students score between: 52.38 and 91.16 (2 Std Devs)
~99.7% of students score between: 42.69 and 100.86 (3 Std Devs)


### Final Data-Driven Statistical Observations

1. **Variability in Academic Readiness:** `Academic_Readiness_Score` exhibits the greatest variability in the dataset with a standard deviation of 9.69 and a massive 56.1-point range. This indicates that while sleep and study hours are fairly consistent across the student body, actual academic preparedness spans a highly diverse spectrum.

2. **The Outlier Pull on Averages:** Identifying and removing 20 outliers from `Monthly_Discretionary_Spending` dropped the Mean by roughly $370 (from $6343.78 down to $5973.72), but barely shifted the Median. This perfectly demonstrates how a small handful of extreme high-spending students artificially skewed the campus average upward.

3. **Statistical Dependence of Jobs and Stress:** Having a part-time job and experiencing high stress are strictly dependent events ($P(A \text{ and } B) \neq P(A) \times P(B)$). The baseline probability of a student having high stress is only 7.5% ($P(B)$), but if a student has a part-time job, their chance of high stress nearly triples to 20.39% ($P(B|A)$).

4. **Employment as a Stress Predictor:** The conditional probability $P(A|B)$ reveals a striking insight: if you look strictly at the students experiencing high stress (Score $\ge$ 7), 68.89% of them have a part-time job. This makes employment status a primary indicator of severe student stress.

5. **Extreme Academic Anomalies (Z-Scores):** The student with the lowest `Academic_Readiness_Score` (41.90) has a Z-score of -3.08. According to the Empirical Rule, 99.7% of all students should score above 42.69. Therefore, this specific student falls in the bottom 0.15% of the curve, representing an extreme statistical anomaly that warrants immediate academic intervention.